# 第 11 章 内生性: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/11-endogeneity.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 200              # 標本の大きさ
correlation = 0.5    # x1 と欠落した x2 の相関
error_sd = 0.5       # x の測定誤差の標準偏差
sd_demand = 2.0      # 需要の誤差の標準偏差
sd_supply = 1.0      # 供給の誤差の標準偏差
repetitions = 500    # 標本を取り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 欠落変数・測定誤差・同時性

In [ ]:
import numpy as np

from lib.chapter11.endogeneity import (
    attenuation_factor,
    draw_omitted,
    ovb_decomposition,
    simulate_measurement_error,
    simulate_omitted,
    simultaneity_slope,
)

y, x1, x2 = draw_omitted(n, correlation, np.random.default_rng(seed))
ovb = ovb_decomposition(y, x1, x2)
print(f"欠落変数: 短い回帰 {ovb.short:.3f}、長い回帰 {ovb.long:.3f}"
      f"（真の値 1）、補助回帰の傾き {ovb.delta:.3f}")

omitted = simulate_omitted(n, correlation, repetitions, np.random.default_rng(seed))
print(omitted.agg(["mean", "std"]).round(3))

noisy = simulate_measurement_error(
    n, error_sd, repetitions, np.random.default_rng(seed)
)
print(f"測定誤差: 傾きの平均 {noisy.mean():.3f}、"
      f"理論の減衰係数 {attenuation_factor(1.0, error_sd**2):.3f}")
print(f"同時性: 傾きの大標本での値 {simultaneity_slope(sd_demand, sd_supply):.3f}")

## 図: 欠落変数の偏り

In [ ]:
from lib.chapter10.plot import plot_estimate_comparison

plot_estimate_comparison(omitted, 1.0)

## 図: 市場（需要と供給の連立）

In [ ]:
from lib.chapter11.endogeneity import draw_market
from lib.chapter11.plot import plot_market

plot_market(draw_market(n, sd_demand, sd_supply, np.random.default_rng(seed)))

## 試してみよう

- `correlation` を 0.9 にすると、x2 を落とした回帰の係数は 1.943 から 2.803
  へ増えます（理論の偏り 2·相関 だけ、真の値 1 から離れます）。x2 を入れた回帰は
  0.944 で、真の値の近くです
- `error_sd` を 1.0 にすると、測定誤差のある回帰の傾きは 0.803 から 0.505 に縮み、
  理論の減衰係数 0.500 に近づきます
- `sd_demand` を 1.0、`sd_supply` を 2.0 にすると、傾きの大標本での値は 0.600 から
  −0.600 になります。需要と供給のどちらの誤差が大きいかで、OLS
  の傾きの符号まで変わります